# 02 — Silver: Sales pipeline core (M5)

**Owner:** M5 • **Tier:** Light • **Phase:** 4 (Thu 14 May lunch – Sun 17 May)

## Scope — POS + Payment + Returns only (EC, DC, T9 moved to LEAD)
- **Source tables (13):**
  - `pos_*` (6): pos_transactions, pos_transaction_lines, pos_cashiers, pos_payment_methods, pos_status_codes, pos_stores
  - `pg_*` (3): pg_transactions, pg_instrument_types, pg_status_codes
  - `rr_*` (4): rr_return_requests, rr_return_receipts, rr_refund_events, rr_return_reasons
- **EC, DC, attribution bridge are now LEAD's** (see notebooks/02_silver_lead.ipynb)
- **Silver transform owned: T8** — Return revenue period (formula, well-defined)
- **Anomalies in your territory:** A1, A2, A3 (3 — A3 is documentation-only)

## ⚠️ Day 2 task
Verify `pg_transactions` date format. UNDERSTANDING.md says Unix epoch; xlsx dictionary says `YYYY-MM-DD HH:MM:SS`. Run:
```sql
SELECT created_ts, TYPEOF(created_ts) FROM NEXAMART_BRONZE.pg_transactions LIMIT 5
```
Update `_shared/utils_dates.py::parse_pg_timestamp()` if needed. Post in team channel.

## T8 — Return revenue period
For partial refunds processed in September for August purchases, produce BOTH columns on `silver_refund_events`:
- `revenue_impact_original_period` (impact assigned to original purchase month)
- `revenue_impact_return_period` (impact assigned to refund month)
Don't choose; let downstream marts choose. Gold default is return_period.

## Hint script
- **A1 (cancelled with revenue):** profile `ec_orders` where `order_status='CANCELLED'`. Does `subtotal_excl_tax > 0`? Sales dashboards probably sum that without filtering. Flag `CANCELLED_WITH_REVENUE`.
- **A2 (payment captured AFTER cancel):** for the cancelled orders, join `pg_transactions` on order id. When were payments captured relative to cancellation timestamp from `ec_order_status_history` (lead's Silver)? Flag with `PAYMENT_AFTER_CANCEL`.
- **A3 (tax/shipping inclusion mismatch):** schema-confirmed. POS uses `total_amount_incl_tax`; EC uses `subtotal_excl_tax`. **Document in report Section 7** — no row-level flag needed.

## Hands-on-of-everything checklist
- [ ] Bronze read
- [ ] Silver write: 13 tables (skip the EC/DC/T9 sections below — those moved to lead)
- [ ] Gold dim: dim_promotion
- [ ] Gold facts: fact_store_sale_line, fact_return_line
- [ ] Anomalies: A1, A2, A3
- [ ] Bus Matrix: 2 fact rows + dim_promotion cells
- [ ] Report: Sections 1, 2, 7 (T8), 8 (A1/A2/A3), 9

**Touchpoints:** you read M2's `silver_customer_master` for customer FKs. M3's `silver_product_master` for product FKs. Lead's `silver_dc_delivery_events` for B7 BOPIS analysis (lead's anomaly, but you may help with PG transaction joins).


## Setup — Install connector + widgets (Free Edition serverless)

_Brief Section 7.4 specified a Spark-Snowflake Maven JAR; Free Edition replaces this with the pure-Python `snowflake-connector-python`. See `docs/databricks_setup.md`._

In [ ]:
%pip install -q snowflake-connector-python pandas rapidfuzz
# dbutils.library.restartPython()  # SKIP on Free Edition — kernel hangs

In [ ]:
dbutils.widgets.text('sf_account',   'rhxendw-yb24678')
dbutils.widgets.text('sf_user',      'NEXAMART_LEAD')   # change to NEXAMART_M{2..6} for members
dbutils.widgets.text('sf_password',  '')                # paste at notebook run time
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role',      'ACCOUNTADMIN')    # NEXAMART_ENGINEER for members

## Cell 2 — Imports & helpers

In [ ]:
from pyspark.sql import functions as F, Window
import sys
sys.path.append('/Workspace/Repos/<your-org>/nexamart-m1/notebooks/_shared')
from utils_dates     import parse_date, is_parse_failure
from utils_keys      import surrogate_key, anonymous_key
from utils_anomaly   import add_anomaly_columns, flag, flag_date_parse_failure
from utils_snowflake import read_from_snowflake, write_to_snowflake

def read_bronze(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_BRONZE')

def read_silver(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_SILVER')

def write_silver(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_SILVER')
    print(f'Wrote silver.{t} ({n} rows)')

def write_gold(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_GOLD')
    print(f'Wrote gold.{t} ({n} rows)')

## Cell 3 — POS Silver: `silver_pos_transactions` + `silver_pos_transaction_lines`

**TODO:**
- Parse `txn_date` from POS (DD/MM/YYYY format — `parse_date(col, 'ddmmyyyy')`)
- Join `pos_status_codes` for `canonical_status` via `silver_status_code_mapping`
- SK on `txn_id` for transactions; on `(txn_id, line_no)` for lines
- A3 awareness: POS `total_amount_incl_tax` is tax-inclusive — note in column docstring; no row-level flag
- Flag DATE_PARSE_FAIL if any
- Write both Silver tables

In [ ]:
# TODO M5: POS silvers

## Cell 4 — A1 / A2 detection (read Lead's `silver_ec_orders`)

> ℹ️ **EC silver build moved to Lead** (`notebooks/02_silver_lead.ipynb`). M5 reads Lead's output to flag A1 + A2; the silver_ec_orders / silver_ec_order_lines tables are not yours to write.

**TODO:**
- Read `silver_ec_orders` (Lead's output, available from Day 5–6)
- **Flag A1:** `(order_status='CANCELLED') AND (subtotal_excl_tax > 0)` → `CANCELLED_WITH_REVENUE`
- **Flag A2 setup:** join `silver_pg_transactions` on order id. For cancelled orders, find captures whose `captured_ts` > the cancellation timestamp from `silver_ec_order_status_history`. Flag with `PAYMENT_AFTER_CANCEL`.
- A3 note: `subtotal_excl_tax` is tax-exclusive (opposite of POS). Document in your report Section 7.
- Write the flag updates back to NEXAMART_SILVER (or to a sibling annotation table — check with lead in standup)

In [ ]:
# TODO M5: EC silvers + A1/A2 detection
# orders = read_bronze('ec_orders')
# orders = orders.withColumn('order_date_parsed', parse_date(F.col('order_date'), 'iso_date'))
# orders = add_anomaly_columns(orders)
# orders = flag(orders,
#     condition=(F.col('order_status') == 'CANCELLED') & (F.col('subtotal_excl_tax') > 0),
#     reason_code='CANCELLED_WITH_REVENUE', status='FLAGGED_ANOMALY', certainty='UNRELIABLE')

## Cell 5 — Payment Silver: `silver_pg_transactions`

**TODO:**
- Read `pg_transactions`
- ⚠️ Use the verified format from your Day 2 check: either `parse_date(col, 'unix_epoch')` or swap `parse_pg_timestamp` to `to_timestamp(col, 'yyyy-MM-dd HH:mm:ss')`
- SK on `transaction_id`
- Apply T2 status normalisation via `silver_status_code_mapping`
- This table feeds A2 detection in Cell 4
- Write

In [ ]:
# TODO M5: pg silver

## ~~Cell 6 — Delivery Silver + A14 detection~~ — MOVED TO LEAD

> ℹ️ DC silver build + A14 (`COURIER_CLOCK_DRIFT`) detection both moved to Lead per the re-allocation. See `notebooks/02_silver_lead.ipynb` Cell for `silver_dc_delivery_events`. M5 only reads it (e.g. for B7 BOPIS gap analysis if you take that on).

In [ ]:
# MOVED TO LEAD — see notebooks/02_silver_lead.ipynb (silver_dc_shipments + silver_dc_delivery_events + A14 detection)

## Cell 7 — Returns/Refunds Silver + T8 (return revenue period)

**TODO:**
- Read `rr_return_requests` (DD-Mon-YYYY format — `parse_date(col, 'ddmonyyyy')`), `rr_return_receipts` (iso_date), `rr_refund_events` (iso_date)
- SK on natural keys
- T8: for each refund event, compute BOTH:
  - `revenue_impact_original_period` (assigned to original purchase month)
  - `revenue_impact_return_period` (assigned to refund month)
- Apply rule: full refund = -1.0 × subtotal; partial refund = -1.0 × refund_amount; restocking fee subtracted from refund (not added to revenue)
- Flag B2 ambiguous cases (cross-period partials) with `REFUND_PARTIAL_PERIOD_AMBIGUITY`, status `FLAGGED_AMBIGUOUS`
- Write all three Silver tables

In [ ]:
# TODO M5: returns/refunds + T8

## ~~Cell 8 — T9: Campaign attribution bridge~~ — MOVED TO LEAD

> ℹ️ T9 (`silver_campaign_attribution_bridge`) moved to Lead per the re-allocation. See `notebooks/02_silver_lead.ipynb` T9 section. M5 doesn't produce or consume this directly.

In [ ]:
# MOVED TO LEAD — see notebooks/02_silver_lead.ipynb (T9 attribution bridge)

## Cell 9 — Acceptance check

Before requesting PR review:
1. `silver_ec_orders` has flagged rows for `CANCELLED_WITH_REVENUE` (you'll see how many)
2. `silver_pg_transactions` × `silver_ec_orders` join surfaces `PAYMENT_AFTER_CANCEL` rows
3. `silver_dc_shipments` has `DELIVERY_BEFORE_SHIP` flagged on a non-trivial number of shipments
4. T8: every `silver_refund_events` row has BOTH `revenue_impact_original_period` AND `revenue_impact_return_period` populated
5. T9: `silver_campaign_attribution_bridge` exists and joins ws_sessions (UTM=BTS2024, ~hundreds) to in-window EC orders
6. B7 BOPIS gap analysis: count BOPIS_COLLECTED events and compare to BOPIS DELIVERED orders — gap is your candidate set

In [ ]:
# Verifications (uncomment after building):
# orders = read_silver('silver_ec_orders')
# orders.groupBy('anomaly_reason_code').count().show(truncate=False)
# 
# # B7 gap analysis
# de = read_silver('silver_dc_delivery_events')
# bopis_collected = de.filter(F.col('event_type') == 'BOPIS_COLLECTED').count()
# bopis_orders_delivered = orders.filter((F.col('delivery_method_code') == 'BOPIS') & (F.col('order_status') == 'DELIVERED')).count()
# print(f'BOPIS_COLLECTED events: {bopis_collected}; BOPIS DELIVERED orders: {bopis_orders_delivered}')
# print(f'Gap (orders without pickup confirmation): {bopis_orders_delivered - bopis_collected}')